# Notebook 00 — Environment and data intake

**Purpose.** Probe the DGX environment, run the train.csv schema gate, inventory and checksum every referenced Parquet file, and estimate cache bytes. **Inputs:** `HMS_DATA_ROOT` (read-only original HMS files). **Outputs:** `private/provenance/{preflight,schema_report,source_manifest}.json`. **Partitions:** none (no labels are modelled here). **GPU:** off. **Budget:** CPU only.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


## Environment probe (no secrets, no environment dump)

In [ ]:
from cape_eeg.status import environment_snapshot
env = environment_snapshot(); print(json.dumps({k: v for k, v in env.items() if k != 'timestamp'}, indent=1))

## Schema gate, file inventory, checksums and byte estimate
The audit fails closed on duplicate `label_id`, zero-vote rows, off-grid offsets or missing files.

In [ ]:
run(['audit_source.py', '--workers', '8'])
sm = read_json(ws.provenance / 'source_manifest.json'); sr = read_json(ws.provenance / 'schema_report.json')
print(json.dumps({k: sm[k] for k in ['n_eeg_files','n_spectrogram_files','source_bytes_total','rows_with_truncated_eeg_window','rows_with_short_spectrogram_context','source_manifest_hash','preprocess_hash']}, indent=1))
print(json.dumps({k: sr[k] for k in ['n_rows','n_patients','n_eeg','n_spectrograms','vote_sum_min','vote_sum_max','n_single_vote','tied_max_rows','consensus_mismatch_unique_rows','spectrogram_offset_source_spelling']}, indent=1))
print('byte estimate:', sm['byte_estimate'])

## Alignment inspection on a synthetic impulse (no patient data)
A 3 Hz burst confined to seconds [22, 28) must land in the foveated target columns [8, 24) and the 3 Hz frequency bin.

In [ ]:
import numpy as np
from cape_eeg.data.spectral import RawSpectralEncoder
from cape_eeg.data.montage import EXPECTED_SOURCE_COLUMNS
from cape_eeg.contracts import foveated_time_edges
enc = RawSpectralEncoder(EXPECTED_SOURCE_COLUMNS); t = np.arange(10000) / 200; x = np.zeros((10000, 20), np.float32)
burst = (t >= 22) & (t < 28); x[burst, :] = np.sin(2 * np.pi * 3 * t[burst])[:, None] * 50
for c in [1, 4, 5, 12, 15, 16]: x[:, c] = 0
r = enc.encode(x, np.ones_like(x, bool)); e = foveated_time_edges(); col = r['foveated'][0].mean(0); hot = np.where(col > col.min() + 5)[0]
print('hot foveated columns', hot.min(), '..', hot.max(), '-> seconds', round(e[hot.min()], 2), '..', round(e[hot.max() + 1], 2))
fb = int(np.argmax(r['foveated'][0][:, 16])); print('peak frequency bin', fb, '->', enc.frequency_edges[fb].round(2), '-', enc.frequency_edges[fb + 1].round(2), 'Hz')
assert hot.min() >= 8 and hot.max() < 24 and 2.5 <= enc.frequency_edges[fb] <= 3.5, 'alignment check failed'
print('alignment check PASS')

## Go / no-go

In [ ]:
led = Ledger(ws.provenance / 'ledger.jsonl').latest(); print({k: v['status'] for k, v in led.items() if k.startswith('I1')})
print('G0 environment:', 'PASS' if env.get('cuda_available') else 'FAIL'); print('G1 source metadata + alignment:', 'PASS' if led.get('I1_source_audit', {}).get('status') == 'PASS' else 'NOT_RUN')
print('source rights: research use of the original Kaggle competition data by the local user; no redistribution of raw data (recorded in DATA_ACCESS.md)')